# ESOL 水溶解度预测
CPU 即可运行。先阅读说明，再选择 Runtime → Run all（运行时 → 全部运行）。安装时间取决于网络；本次训练及出图耗时 9.13 秒。

数据来源：DeepChem ESOL；原始文献 DOI:10.1021/ci034243x。结果是回归误差，不是分类准确率。

In [ ]:
%pip -q install numpy==2.3.5 pandas==2.2.3 scikit-learn==1.8.0 matplotlib==3.10.8 rdkit==2026.9.2


## 写入随附的公开数据与完整代码
CSV 和脚本已内嵌，避免下载地址失效。

In [ ]:
from pathlib import Path
import base64
p = Path("/content/esol_project") if Path("/content").exists() else Path("esol_project")
(p/"data").mkdir(parents=True,exist_ok=True)
(p/"data/delaney-processed.csv").write_bytes(base64.b64decode('...'))
(p/"run_project.py").write_text('"""CPU-only ESOL benchmark. Run: python run_project.py"""\nfrom pathlib import Path\nimport hashlib, itertools, json, platform, time, sys\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom rdkit import Chem, rdBase\nfrom rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors\nfrom rdkit.Chem.Scaffolds import MurckoScaffold\nfrom sklearn import __version__ as sklearn_version\nfrom sklearn.dummy import DummyRegressor\nfrom sklearn.linear_model import Ridge\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score\nfrom sklearn.base import clone\n\nROOT = Path(__file__).resolve().parent\nOUT = ROOT / \'results\'\nOUT.mkdir(exist_ok=True)\nSEED = 42\nFEATURES = [\'MolWt\',\'LogP\',\'RotatableBonds\',\'AromaticProportion\',\'HBD\',\'HBA\',\'TPSA\',\'RingCount\',\'HeavyAtoms\',\'FractionCSP3\']\nTARGET = \'measured log solubility in mols per litre\'\n\ndef features(m):\n    heavy = m.GetNumHeavyAtoms()\n    return [Descriptors.MolWt(m), Crippen.MolLogP(m), Lipinski.NumRotatableBonds(m),\n            sum(a.GetIsAromatic() for a in m.GetAtoms()) / max(heavy, 1),\n            Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m), rdMolDescriptors.CalcTPSA(m),\n            rdMolDescriptors.CalcNumRings(m), heavy, rdMolDescriptors.CalcFractionCSP3(m)]\n\ndef metrics(y, p):\n    return dict(RMSE=float(np.sqrt(mean_squared_error(y,p))), MAE=float(mean_absolute_error(y,p)), R2=float(r2_score(y,p)))\n\ndef split_indices(frame, mode):\n    ids = np.arange(len(frame))\n    if mode == \'random\':\n        tr, rest = train_test_split(ids, test_size=.30, random_state=SEED)\n        va, te = train_test_split(rest, test_size=.5, random_state=SEED)\n    else:\n        # Greedy, deterministic 70/15/15 scaffold grouping; all acyclic molecules\n        # share the empty scaffold. Fractions are approximate, not exact.\n        groups = [list(v) for v in frame.groupby(\'scaffold\',sort=True).indices.values()]\n        groups.sort(key=lambda g:(-len(g),min(g)))\n        tr, va, te = [], [], []\n        for group in groups:\n            if len(tr)+len(group) <= .70*len(frame): tr.extend(group)\n            elif len(va)+len(group) <= .15*len(frame): va.extend(group)\n            else: te.extend(group)\n        tr, va, te = map(np.array,(tr,va,te))\n        sets=[set(frame.iloc[idx].scaffold) for idx in (tr,va,te)]\n        assert not (sets[0]&sets[1] or sets[0]&sets[2] or sets[1]&sets[2])\n    assert len(set(tr)&set(va)) == len(set(tr)&set(te)) == len(set(va)&set(te)) == 0\n    assert len(tr)+len(va)+len(te) == len(frame)\n    return tr, va, te\n\ndef main():\n    start = time.perf_counter()\n    source = ROOT/\'data/delaney-processed.csv\'\n    if not source.exists():\n        import urllib.request\n        source.parent.mkdir(exist_ok=True)\n        urllib.request.urlretrieve(\'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv\',source)\n    raw = pd.read_csv(source)\n    records=[]\n    invalid=0\n    for _, row in raw.iterrows():\n        mol = Chem.MolFromSmiles(str(row[\'smiles\']).strip())\n        if mol is None or not np.isfinite(row[TARGET]):\n            invalid += 1\n            continue\n        records.append({\'canonical_smiles\':Chem.MolToSmiles(mol), \'logS\':float(row[TARGET])})\n    data = pd.DataFrame(records).groupby(\'canonical_smiles\',sort=True,as_index=False).agg(logS=(\'logS\',\'mean\'),measurements=(\'logS\',\'size\'))\n    mols = [Chem.MolFromSmiles(s) for s in data.canonical_smiles]\n    data[\'scaffold\'] = [MurckoScaffold.MurckoScaffoldSmiles(mol=m,includeChirality=False) for m in mols]\n    X = np.array([features(m) for m in mols]); y = data.logS.to_numpy()\n    assert np.isfinite(X).all()\n    # CSV-provided predicted ESOL values and descriptors are NEVER used as features.\n    pd.concat([data,pd.DataFrame(X,columns=FEATURES)],axis=1).to_csv(OUT/\'clean_features.csv\',index=False)\n    metadata = {\'raw_rows\':len(raw),\'invalid_rows\':invalid,\'unique_molecules\':len(data),\n                \'duplicate_rows_merged\':len(records)-len(data),\'seed\':SEED,\n                \'source_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\n                \'versions\':{\'python\':platform.python_version(),\'rdkit\':rdBase.rdkitVersion,\'sklearn\':sklearn_version,\'numpy\':np.__version__,\'pandas\':pd.__version__},\'splits\':{}}\n    all_scores=[]; trials=[]; predictions=[]\n    for mode in [\'scaffold\',\'random\']:\n        tr,va,te=split_indices(data,mode)\n        assign=data[[\'canonical_smiles\',\'scaffold\',\'logS\']].copy()\n        assign[\'split\']=\'\'; assign.loc[tr,\'split\']=\'train\'; assign.loc[va,\'split\']=\'validation\'; assign.loc[te,\'split\']=\'test\'\n        assign.to_csv(OUT/f\'{mode}_split.csv\',index=False)\n        best_score=float(\'inf\'); best_params=None\n        # All 12 candidate models see TRAIN only; validation selects parameters.\n        for depth,leaf,mf in itertools.product([None,8],[1,2,4],[1.0,.7]):\n            params=dict(n_estimators=200,max_depth=depth,min_samples_leaf=leaf,max_features=mf,random_state=SEED,n_jobs=2)\n            model=RandomForestRegressor(**params).fit(X[tr],y[tr])\n            score=metrics(y[va],model.predict(X[va]))[\'RMSE\']\n            trials.append({\'split\':mode,\'max_depth\':str(depth),\'min_samples_leaf\':leaf,\'max_features\':mf,\'validation_RMSE\':score})\n            if score < best_score: best_score,best_params=score,params\n        metadata[\'splits\'][mode]={\'train\':len(tr),\'validation\':len(va),\'test\':len(te),\n            \'best_params\':best_params,\'best_validation_RMSE\':best_score,\n            \'scaffold_overlap_train_test\':len(set(data.iloc[tr].scaffold)&set(data.iloc[te].scaffold))}\n        configs=[(\'Mean\',DummyRegressor(),10),(\'Ridge4\',make_pipeline(StandardScaler(),Ridge(alpha=1.0)),4),\n                 (\'Ridge10\',make_pipeline(StandardScaler(),Ridge(alpha=1.0)),10),\n                 (\'RF_default\',RandomForestRegressor(n_estimators=200,random_state=SEED,n_jobs=2),10),\n                 (\'RF_tuned\',RandomForestRegressor(**best_params),10)]\n        fig,ax=plt.subplots(figsize=(5.8,5.2))\n        for name,model,k in configs:\n            # No test data are used for fitting, scaling, or parameter selection.\n            dev=np.r_[tr,va]; model.fit(X[dev,:k],y[dev]); pred=model.predict(X[te,:k])\n            score=metrics(y[te],pred)\n            all_scores.append({\'split\':mode,\'model\':name,**score})\n            for ix,p in zip(te,pred): predictions.append({\'split\':mode,\'model\':name,\'canonical_smiles\':data.iloc[ix].canonical_smiles,\'actual\':y[ix],\'predicted\':p})\n            print(mode,name,{k:round(v,4) for k,v in score.items()},flush=True)\n            if name==\'RF_tuned\':\n                ax.scatter(y[te],pred,s=18,alpha=.65,color=\'#176B78\')\n                pd.DataFrame({\'feature\':FEATURES,\'impurity_importance\':model.feature_importances_}).sort_values(\'impurity_importance\',ascending=False).to_csv(OUT/f\'{mode}_feature_importance.csv\',index=False)\n        ax.plot([-12,2],[-12,2],\'--\',color=\'#555555\',linewidth=1)\n        ax.set(xlabel=\'Measured logS (mol/L)\',ylabel=\'Predicted logS (mol/L)\',title=f\'Tuned random forest: {mode} test\')\n        fig.tight_layout();fig.savefig(OUT/f\'{mode}_parity.png\',dpi=190);plt.close(fig)\n    scores=pd.DataFrame(all_scores);scores.to_csv(OUT/\'metrics.csv\',index=False)\n    pd.DataFrame(trials).to_csv(OUT/\'tuning_trials.csv\',index=False)\n    preds=pd.DataFrame(predictions);preds.to_csv(OUT/\'test_predictions.csv\',index=False)\n    # Paired bootstrap of test molecules; conditional on each fixed split/model.\n    cis=[]\n    for mode in [\'scaffold\',\'random\']:\n        a=preds[(preds.split==mode)&(preds.model==\'RF_default\')].reset_index(drop=True)\n        b=preds[(preds.split==mode)&(preds.model==\'RF_tuned\')].reset_index(drop=True)\n        assert a.canonical_smiles.equals(b.canonical_smiles)\n        rng=np.random.default_rng(SEED); values=[]\n        for _ in range(1000):\n            idx=rng.integers(0,len(a),len(a))\n            values.append(np.sqrt(np.mean((b.actual.to_numpy()[idx]-b.predicted.to_numpy()[idx])**2))-np.sqrt(np.mean((a.actual.to_numpy()[idx]-a.predicted.to_numpy()[idx])**2)))\n        cis.append({\'split\':mode,\'delta_RMSE_tuned_minus_default_low\':float(np.quantile(values,.025)),\'delta_RMSE_tuned_minus_default_high\':float(np.quantile(values,.975))})\n    pd.DataFrame(cis).to_csv(OUT/\'paired_bootstrap.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(8.6,4.6))\n    names=scores.model.unique();xx=np.arange(len(names))\n    for offset,mode,color in [(-.18,\'scaffold\',\'#176B78\'),(.18,\'random\',\'#D98B35\')]:\n        vals=scores[scores.split==mode].set_index(\'model\').loc[names,\'RMSE\']\n        bars=ax.bar(xx+offset,vals,.34,label=mode,color=color)\n        ax.bar_label(bars,fmt=\'%.2f\',fontsize=8,padding=3)\n    ax.set_xticks(xx,names);ax.set(ylabel=\'Test RMSE (log mol/L)\',title=\'Model comparison on two fixed partitions\')\n    ax.legend(frameon=False);ax.spines[[\'top\',\'right\']].set_visible(False)\n    fig.tight_layout();fig.savefig(OUT/\'model_comparison.png\',dpi=190);plt.close(fig)\n    fig,ax=plt.subplots(figsize=(6,3.5));ax.hist(y,bins=30,color=\'#176B78\',edgecolor=\'white\')\n    ax.set(xlabel=\'Measured logS (mol/L)\',ylabel=\'Molecules\',title=\'ESOL target distribution after deduplication\')\n    fig.tight_layout();fig.savefig(OUT/\'target_distribution.png\',dpi=190);plt.close(fig)\n    metadata[\'runtime_seconds\']=round(time.perf_counter()-start,2)\n    (OUT/\'run_metadata.json\').write_text(json.dumps(metadata,indent=2))\n    print(\'Completed in\',metadata[\'runtime_seconds\'],\'seconds. Results:\',OUT,flush=True)\n\nif __name__==\'__main__\': main()\n',encoding="utf-8")


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(p/"run_project.py")], check=True)


In [ ]:
import csv
from IPython.display import display, Image, Markdown
with open(p/"results/metrics.csv", newline="") as f:
    rows=list(csv.reader(f))
display(Markdown("|"+"|".join(rows[0])+"|\n|"+"|".join(["---"]*len(rows[0]))+"|\n"+"\n".join("|"+"|".join(r)+"|" for r in rows[1:])))
display(Image(filename=str(p/"results/model_comparison.png")))
display(Image(filename=str(p/"results/scaffold_parity.png")))


In [ ]:
import shutil
archive=shutil.make_archive(str(p.parent/"ESOL_my_results"),"zip",p)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Results saved:",archive)
